# Image Generation with Pre-trained Models (Stable Diffusion + KerasCV)
**Author:** [Your Name]  
**Task 02 - Generative AI**

Goal: use a pre-trained text-to-image model (Stable Diffusion) to create images from text prompts, understand how it works, and explore the parameters that control it.

> Run on a GPU runtime (Colab: Runtime -> Change runtime type -> T4 GPU). On CPU it is extremely slow.

## 1. Setup

In [ ]:
!pip install tensorflow keras_cv --upgrade --quiet

In [ ]:
import os, time
import keras_cv
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from PIL import Image

os.makedirs("../outputs", exist_ok=True)

def plot_images(images, titles=None):
    plt.figure(figsize=(20, 20))
    for i in range(len(images)):
        ax = plt.subplot(1, len(images), i + 1)
        plt.imshow(images[i])
        if titles: plt.title(titles[i], fontsize=14)
        plt.axis("off")
    plt.show()

def save_images(images, prefix):
    for i, img in enumerate(images):
        Image.fromarray(img).save(f"../outputs/{prefix}_{i}.png")

## 2. Theory in brief
Stable Diffusion is a **latent diffusion model** made of three parts:
1. **Text encoder** (frozen CLIP language model): prompt -> embedding vectors.
2. **Diffusion model** (U-Net): repeatedly *denoises* a 64x64 latent, guided by the text embeddings via cross-attention (default 50 steps).
3. **Decoder** (VAE): 64x64 latent -> 512x512 image.

Training: add noise to images and teach the network to predict that noise. Generation: start from pure noise and denoise step by step. Prompt control comes from *conditioning* on text and **classifier-free guidance**:  
`eps = eps_uncond + s * (eps_cond - eps_uncond)`, where `s` is the guidance scale (default 7.5).  
See `docs/theory.md` for the full explanation.

## 3. Load the pre-trained model

In [ ]:
model = keras_cv.models.StableDiffusion(img_width=512, img_height=512)

## 4. First image: a simple prompt

In [ ]:
images = model.text_to_image("photograph of an astronaut riding a horse", batch_size=3)
plot_images(images)
save_images(images, "01_astronaut_horse")

## 5. Prompt engineering: a detailed prompt
Adding style, medium and quality keywords changes the result a lot.

In [ ]:
images = model.text_to_image(
    "cute magical flying dog, fantasy art, "
    "golden color, high quality, highly detailed, elegant, sharp focus, "
    "concept art, character concepts, digital painting, mystery, adventure",
    batch_size=3,
)
plot_images(images)
save_images(images, "02_flying_dog")

## 6. Experiment A: number of denoising steps
Same seed and prompt, different `num_steps`.

In [ ]:
prompt = "a cozy wooden cabin in a snowy forest at sunset, highly detailed, digital painting"
step_list = [5, 15, 30, 50]
imgs = [model.text_to_image(prompt, batch_size=1, num_steps=s, seed=42)[0] for s in step_list]
plot_images(imgs, [f"{s} steps" for s in step_list])
save_images(imgs, "03_steps")

## 7. Experiment B: guidance scale
Low = creative / loose, high = strict prompt following.

In [ ]:
scales = [1, 4, 7.5, 15]
imgs = [model.text_to_image(prompt, batch_size=1, unconditional_guidance_scale=g, seed=42)[0] for g in scales]
plot_images(imgs, [f"guidance {g}" for g in scales])
save_images(imgs, "04_guidance")

## 8. Experiment C: seed (reproducibility)
Same seed -> same image; different seed -> different image.

In [ ]:
seeds = [1, 1, 2, 3]
imgs = [model.text_to_image("a red vintage car on a beach, photograph", batch_size=1, seed=s)[0] for s in seeds]
plot_images(imgs, [f"seed {s}" for s in seeds])
save_images(imgs, "05_seeds")

## 9. Experiment D: negative prompt

In [ ]:
base = "portrait of an old fisherman, realistic photograph"
a = model.text_to_image(base, batch_size=1, seed=7)[0]
b = model.text_to_image(base, negative_prompt="blurry, low quality, cartoon, deformed", batch_size=1, seed=7)[0]
plot_images([a, b], ["no negative prompt", "with negative prompt"])
save_images([a, b], "06_negative_prompt")

## 10. More creative prompts

In [ ]:
prompts = [
    "A cute otter in a rainbow whirlpool holding shells, watercolor",
    "An avocado armchair",
    "Teddy bears conducting machine learning research",
]
for i, p in enumerate(prompts):
    imgs = model.text_to_image(p, batch_size=3)
    print(p); plot_images(imgs); save_images(imgs, f"07_prompt{i}")

## 11. Speed optimisations
- **Mixed precision**: compute in `float16`, keep weights in `float32`.
- **XLA (`jit_compile=True`)**: compiles the graph into fast fused kernels.

We benchmark standard vs. optimised models.

In [ ]:
keras.backend.clear_session()
results = []
test_prompt = "A cute otter in a rainbow whirlpool holding shells, watercolor"

# Standard
model = keras_cv.models.StableDiffusion()
model.text_to_image("warm up", batch_size=3)
t = time.time(); model.text_to_image(test_prompt, batch_size=3); results.append(["Standard", time.time() - t])
keras.backend.clear_session()

# Mixed precision + XLA
keras.mixed_precision.set_global_policy("mixed_float16")
model = keras_cv.models.StableDiffusion(jit_compile=True)
model.text_to_image("warm up", batch_size=3)
t = time.time(); imgs = model.text_to_image(test_prompt, batch_size=3); results.append(["XLA + Mixed Precision", time.time() - t])
plot_images(imgs)

print("{:<24}{}".format("Model", "Runtime (s)"))
for n, r in results: print("{:<24}{:.2f}".format(n, r))

In [ ]:
plt.bar([r[0] for r in results], [r[1] for r in results], color=["#888", "#2a6fdb"])
plt.ylabel("Seconds for 3 images"); plt.title("Inference speed")
plt.savefig("../outputs/08_benchmark.png", dpi=150); plt.show()

## 12. Observations / Conclusion
- Stable Diffusion turns text into images by denoising a latent starting from noise, conditioned on a CLIP text embedding.
- More steps improve quality up to a point (~30-50); very few steps give blurry/noisy results.
- Guidance scale trades creativity vs. prompt faithfulness; 7-8 is a good default.
- A fixed seed makes results reproducible; negative prompts remove unwanted traits.
- Mixed precision + XLA make inference noticeably faster on GPU.
- Limitations: bias from training data, trouble with text/hands/counting, and licence/ethical constraints (CreativeML Open RAIL-M).

*Replace these observations with what you actually see in your own outputs.*